# Chapter 15 Notes 

## Determining Optimal Substructure

## Tail recursive
- Tail recursive means the recursive call is the last operation in the function

## Activity Selection: Easy Optimal-Substructure Proof

### Setup
- Let `S_ij` be the set of activities that:
  - start after activity `a_i` finishes, and
  - finish before activity `a_j` starts.
- Let `A_ij` be an optimal (maximum-size) compatible subset of `S_ij`.
- Suppose `a_k` is an activity inside `A_ij`.

Then activities in `A_ij` split into:
- activities before `a_k` (call this `A_ik`),
- activity `a_k`,
- activities after `a_k` (call this `A_kj`).

So:
$$
A_{ij} = A_{ik} \cup \{a_k\} \cup A_{kj}
$$

and

$$
|A_{ij}| = |A_{ik}| + 1 + |A_{kj}|.
$$

### Argument
To show optimal substructure, prove `A_ik` and `A_kj` are each optimal for their own subproblems.

Assume `A_kj` is **not** optimal for `S_kj`.  
Then there exists another compatible set `A'_kj` in `S_kj` with:
\[
|A'_{kj}| > |A_{kj}|.
\]

Now replace `A_kj` in `A_ij` with `A'_kj`.  
The new solution size becomes:
\[
|A_{ik}| + 1 + |A'_{kj}| > |A_{ik}| + 1 + |A_{kj}| = |A_{ij}|.
\]

This contradicts that `A_ij` was optimal.  
So `A_kj` must be optimal.  
By the same argument, `A_ik` must also be optimal.

Therefore, the activity-selection problem has **optimal substructure**.

### DP recurrence
Let `c[i][j]` be the size of an optimal solution for `S_ij`:
\[
c[i][j] =
\begin{cases}
0, & \text{if } S_{ij}=\emptyset \\
\max_{k \in S_{ij}} \left(c[i][k] + 1 + c[k][j]\right), & \text{otherwise}
\end{cases}
\]

In [ ]:
# Tail recursive
def f(n, acc=1):
    if n == 0:
        return acc
    return f(n - 1, acc * n)

In [ ]:
# Non-tail recursive
def f(n):
    if n == 0:
        return 1
    return n * f(n - 1)

- You can transform a tail-recursive procedure to an iterative form.
- Some compilers for certain programming languages perform this task automatically

Because tail recursion carries all needed state in parameters, so each call is just “update state and repeat.”

That maps directly to a loop:

Recursive step: call same function with new arguments.
Iterative step: assign new values to variables and continue while.

In [ ]:
# tail-recursive
def f(state):
    if done(state):
        return result(state)
    return f(next_state(state))

In [ ]:
# iterative
def f(state):
    while not done(state):
        state = next_state(state)
    return result(state)


- No pending work is left after the recursive call so no call stack is required beyond current state

## Activity Selector
- Both iterative and recursive greedy activity selectors run in O(n) time.

## Elements of the greedy Strategy
- Greedy algorithm uses a heuristic strategy of makign the choice that seems the best at the moment
- Sometimems it works, sometimes it doesn't
1. Determine the optimal substructure of the problem
2. Develop a recursive solution
3. Show that if you make the greedy choice, then only one subproblem remains
4. Prove that it is always safe to make the greedy choice (3 and 4 can occur in either order)
5. Convert the recursive algorithm to an iterative algorithm

### Alternatively:
1. Cast the optimization problem as one in which you make a choice and are left with one subproblem to solve.
2. Prove that there is always an optimal solution to the original problem that makes the greedy choice, so that the greedy choice is always safe.
3. Demonstrate optimal substructure by showing that, having made the greedy
choice, what remains is a subproblem with the property that if you combine an optimal solution to the subproblem with the greedy choice you have made, you arrive at an optimal solution to the original problem. 

## Greedy vs Dynamic Programming
- Unlike dynamic programming, which solves the subproblems before making the frst choice, a greedy algorithm makes its ûrst choice before solving any subproblems. A dynamic-programming algorithm proceeds bottom up, whereas a greedy strategy usually progresses top down, making one greedy choice after another, reducing each given problem instance to a smaller one. 
- Proving optimal substructure uses induction on the subproblems to prove making the greedy choice at every step produces an optimal solution
- 0-1 Knapsack (Greedy fails, DP works):
You must take each item whole or not at all. Picking by best value/weight first can block a better combination later. In the example, taking item 1 first (best ratio) is suboptimal; items 2+3 give a better total value under the 50-pound limit.
Reason: local best choice does not always lead to global best, so greedy-choice property fails.
Use dynamic programming.

- Fractional Knapsack (Greedy works):
You can take fractions of items. Taking as much as possible of the highest value/weight item first is always safe, then move to the next best ratio. In the same example setup, this ratio-based greedy order is optimal.
Reason: exchange argument holds; greedy-choice property is valid here.
Use greedy algorithm.

## Huffman Tree Construction (CLRS Pseudocode -> This Code)

### Big picture
The code builds a Huffman tree from character frequencies, then uses that tree to:
1. create binary codes for each character,
2. compress a file into bits,
3. decompress it back.

---

### Step 1: Build frequency table (`C`)
- Read input file one character at a time.
- Count how often each character appears.
- Add a special EOF character with frequency 1.

In code: `compute_freq()`  
Output: `freq_dict` (this is `C` in CLRS).

---

### Step 2: Initialize min-priority queue (`Q = C`)
- Create a min-heap priority queue.
- For each `(char, freq)` in `freq_dict`, insert a `HuffmanNode`.

In code: start of `construct_tree(freq_dict)`.

---

### Step 3: Repeat `n - 1` times (core CLRS loop)
Each iteration:
1. Extract the two minimum-frequency nodes: `x`, `y`.
2. Create a new internal node `z`.
3. Set:
   - `z.left = x`
   - `z.right = y`
   - `z.freq = x.freq + y.freq`
4. Insert `z` back into the min-heap.

In code:
- `x = queue.extract_min()`
- `y = queue.extract_min()`
- `queue.insert(HuffmanNode(freq=x.freq + y.freq, left=x, right=y))`

This exactly matches CLRS `HUFFMAN(C)` lines 4–10.

---

### Step 4: Return the final node as root
- After `n-1` merges, only one node remains in the heap.
- That node is the root of the Huffman tree.

In code: `return queue.extract_min()`

---

## What happens after tree construction

### `encode_chars()`
- Traverse the tree:
  - left edge = bit `0`
  - right edge = bit `1`
- Each leaf gets its Huffman code.

### `compress()`
- Read each input character.
- Replace it with its Huffman bits.
- Write bits to compressed output.
- Append EOF code at the end.

### `decompress()`
- Read bits from compressed file.
- Walk the tree (0 left, 1 right).
- On reaching a leaf:
  - EOF leaf -> stop
  - otherwise write that character and continue.

---

## Complexity (tree construction)
- Time: `O(n log n)` where `n` = number of distinct characters.
- Space: `O(n)`.

### Cost of a Huffman code tree

For a code tree \(T\), its cost is:

$$
B(T) = \sum_{c \in C} c.\text{freq} \cdot d_T(c)
$$

- $C$: set of characters (symbols)
- $c.\mathrm{freq}$: frequency of character $c$
- $d_T(c)$: depth of $c$'s leaf in tree $T$ (code length in bits)

**Meaning:**  
\(B(T)\) is the total weighted path length, i.e., the total number of bits needed to encode the text (up to the same constant factors for all trees).  
A better Huffman tree is one that **minimizes \(B(T)\)**.


## Furthest-in-future
- Used to solve offline cachine problem - if you know in advance the entire sequence of n requests and cache size k, with a goal of minimizing the total number of cache misses, what blocks do you keep in the cache to minimize cache misses
- furthest-in-future chooses to evict the block in the cache whose next access in the request sequence comest furthest in the future
- we can use the number of cache misses produced by an optimal algorithm as a baseline for comparing how well online algorithms perform. 

### Furthest-in-futre optimal offline caching greedy-choice property proof

- Intuition first: If two blocks are currently in cache, and one of them is needed sooner than the other, you should keep the one needed sooner.
So if you must evict one, evict the one needed latest.

Assume:

At request i, cache is full and we miss on block b_i.
z is the cached block used farthest in the future.
S is an optimal solution, but at this moment S evicts some other block x (not z).
We build a new solution S':

Same as S, except at this moment S' evicts z instead of x.
After that, S' tries to mimic S as much as possible.
Now compare S and S' from request i onward:

Right after eviction, caches differ in only one block (x vs z).
Until the next time z is requested, z is not needed, by definition.
So evicting z cannot hurt before that point.
If a difference causes one strategy to miss and the other to hit, it can only be because of x/z.
Since x is used no later than z, any possible extra miss for S' at z is balanced by an earlier hit at x (or no worse outcome).
By the time z is requested, we can make the caches line up again and then follow S exactly.
Conclusion: S' has no more misses than S.
Since S was optimal, S' is also optimal.
And S' uses the greedy choice (evict farthest-in-future) at request i.

Therefore, the greedy choice is safe and belongs to some optimal solution